In [1]:
# Import the libraries required for the RAG pipeline

from pathlib import Path

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

import faiss
import numpy as np

/home/nineleaps/mj_projects/jupyter_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Extract text from the PDF document

pdf_path = next(Path("sample_data").glob("*.pdf"))

reader = PdfReader(pdf_path)

documents = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()

    if text:
        documents.append({
            "page": page_number,
            "text": text
        })

print("PDF:", pdf_path.name)
print("Number of pages:", len(documents))

PDF: Prompt Creation and Failure Pattern Identification (2).pdf
Number of pages: 6


In [3]:
# Split the extracted PDF text into smaller chunks

chunk_size = 1000
chunk_overlap = 200

chunks = []

for document in documents:
    text = document["text"]
    page_number = document["page"]

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "page": page_number,
                "text": chunk_text
            })

        start += chunk_size - chunk_overlap

print("Number of chunks:", len(chunks))

Number of chunks: 23


In [4]:
# Preview the first few text chunks

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i + 1} | Page {chunk['page']} ---")
    print(chunk["text"])


--- Chunk 1 | Page 1 ---
Financial
 
Dataset
 
Curation
 
&
 
Prompt
 
Creation
 
Guidelines
 
1.
 
Purpose
 
&
 
Objectives
 
The
 
purpose
 
of
 
this
 
project
 
is
 
to
 
curate
 
and
 
generate
 
high-quality
 
financial
 
dataset
 
items
 
that
 
test
 
whether
 
an
 
AI
 
model
 
can
 
perform
 
realistic,
 
analyst-level
 
financial
 
work.
 
Each
 
item
 
must
 
represent
 
a
 
real
 
professional
 
workflow
 
rather
 
than
 
a
 
textbook
 
exercise,
 
simple
 
data
 
extraction
 
task,
 
or
 
generic
 
financial
 
explanation.
 
The
 
workflow
 
forces
 
the
 
model
 
to
 
read
 
multiple
 
documents,
 
extract
 
relevant
 
financial
 
data,
 
resolve
 
discrepancies,
 
apply
 
custom
 
financial
 
methodologies,
 
perform
 
multi-step
 
cascading
 
calculations,
 
and
 
present
 
professional,
 
objectively
 
gradable
 
outputs.
 
To
 
"hill-climb"
 
and
 
improve
 
performance,
 
we
 
need
 
to
 
feed
 
the
 
model
 
training
 
examples
 
that
 
reflect
 
the
 
complexity


In [15]:
# Load the Gemini API key from the .env file

import os
from dotenv import load_dotenv

load_dotenv()

print("API key available:", bool(os.getenv("GEMINI_API_KEY")))

API key available: True


In [8]:
# Create the Gemini client using the API key from the environment

import os
from google import genai

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

# Generate embeddings using Gemini

embedding_results = client.models.embed_content(
    model="gemini-embedding-001",
    contents=chunk_text
)

embeddings = np.array(
    [embedding.values for embedding in embedding_results.embeddings],
    dtype="float32"
)

print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

Number of embeddings: 1
Embedding dimensions: 3072


In [9]:
# Generate one embedding for each text chunk

embeddings = []

for chunk in chunks:
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=chunk["text"]
    )

    embeddings.append(response.embeddings[0].values)

embeddings = np.array(
    embeddings,
    dtype="float32"
)

print("Number of chunks:", len(chunks))
print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

Number of chunks: 23
Number of embeddings: 23
Embedding dimensions: 3072


In [10]:
# Store the embeddings in a FAISS vector database

import faiss

embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dimension)

index.add(embeddings)

print("Number of vectors stored:", index.ntotal)

Number of vectors stored: 23


In [11]:
# Retrieve the most relevant chunks for a question

question = "What is the main topic of this document?"

question_response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=question
)

question_embedding = np.array(
    [question_response.embeddings[0].values],
    dtype="float32"
)

distances, indices = index.search(
    question_embedding,
    k=3
)

print("Retrieved chunk indices:", indices[0])
print("Distances:", distances[0])

Retrieved chunk indices: [ 9 12  8]
Distances: [0.7612554 0.7768558 0.7949055]


In [12]:
# Display the retrieved PDF chunks

retrieved_chunks = []

for i in indices[0]:
    retrieved_chunks.append(chunks[i])

for number, chunk in enumerate(retrieved_chunks, start=1):
    print(f"\n--- Retrieved Chunk {number} | Page {chunk['page']} ---")
    print(chunk["text"])


--- Retrieved Chunk 1 | Page 3 ---
a
 
tables
 
formatted
 
within
 
PDFs,
 
including
 
debt
 
schedules,
 
mortgage
 
collateral
 
tables,
 
budget/actuals
 
tables,
 
return
 
history
 
tables,
 
option
 
grids,
 
factor
 
return
 
datasets,
 
and
 
cash-flow
 
schedules.
 
Unacceptable
 
Sources
 
●
 
Live
 
URLs,
 
paywalled
 
sources,
 
or
 
unstable
 
web
 
pages.
 
●
 
Screenshots
 
or
 
image-only
 
documents
 
without
 
extractable
 
text.
 
●
 
Wikipedia
 
pages,
 
generic
 
research
 
reports,
 
or
 
textbook-style
 
educational
 
material.
 
●
 
Documents
 
that
 
render
 
the
 
answer
 
solvable
 
from
 
a
 
single,
 
obvious
 
summary
 
table.
 
Complexity
 
Metrics
 
&
 
Curation
 
Thresholds
 
Human
 
raters
 
must
 
align
 
new
 
examples
 
with
 
the
 
following
 
empirical
 
complexity
 
target
 
ranges:
 
●
 
Prompt
 
Length:
 
Must
 
be
 
between
 
2,000
 
and
 
4,000
 
characters
 
(roughly
 
400
 
to
 
800
 
words).

--- Retrieved Chunk 2 | Page 4 ---
the
 
exp

In [16]:
# Ask the user for a question and generate an answer from the PDF

question = input("Enter your question: ")

# Generate an embedding for the question
question_response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=question
)

question_embedding = np.array(
    [question_response.embeddings[0].values],
    dtype="float32"
)

# Retrieve the 3 most relevant chunks
distances, indices = index.search(
    question_embedding,
    k=3
)

retrieved_chunks = [
    chunks[i] for i in indices[0]
]

# Combine retrieved chunks into context
context = "\n\n".join(
    chunk["text"] for chunk in retrieved_chunks
)

# Create the RAG prompt
prompt = f"""
Answer the question using only the context provided below.

If the answer is not present in the context, say:
"I could not find the answer in the document."

Context:
{context}

Question:
{question}

Answer:
"""

# Generate the final answer
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print("\nAnswer:")
print(response.text)

Enter your question:  Who is the first President of India



Answer:
I could not find the answer in the document.
